# Swing Vision Transformation

#### Converting SwingVision data into our format

#### Trying to find other swingvision data:

- https://bubbletao.com/2019/03/31/tennis-score-tracking-and-exported-data-analysis-with-apple-watch-using-swing-app/  (no data, but walks through common errors and fixes in data)


# Comparing website manually tagged data



### Load in Data

In [ ]:
import pandas as pd
import numpy as np
import os 
import re

# Option to display max rows/columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
your_file_name = ''
clean_df = pd.read_csv(your_file_name

In [ ]:
clean_df.shape

In [ ]:
clean_df.head(20)

# Notebook Start

### Load in data

In [ ]:
import pandas as pd
import numpy as np
import os 
import re

# Option to display max rows/columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# Input file name here
your_file_name = 'EliseWagle_TexasTech_OriginalScore.xlsx'
swing_data = pd.read_excel(your_file_name, sheet_name='Shots')
swing_data.shape

In [ ]:
swing_data.head()

### Clean swing vision data

In [ ]:
swing_data.query('Stroke == "Feed"').head()

In [ ]:
swing_data.query('Shot == 0').head()

In [ ]:
swing_data.query('Type == "none"').head()

#### Remove all the rows where it is a feed

In [ ]:
rows_to_drop = swing_data.query('Type == "none"').index
swing_data = swing_data.drop(rows_to_drop)
# Important to reindex to avoid missing indicies
swing_data = swing_data.reset_index(drop=True)
swing_data.shape

In [ ]:
swing_data.head()

### Load in Points data

In [ ]:
your_file_name = 'EliseWagle_TexasTech_OriginalScore.xlsx'
swing_data_points = pd.read_excel(your_file_name, sheet_name='Points')
swing_data_points.shape

In [ ]:
def create_point(server, player1score, player2score):
    if server == "host":
        return str(player1score) + "-" + str(player2score)
    else:
        return str(player2score) + "-" + str(player1score)
    
swing_data_points['pointScore'] = swing_data_points.apply(lambda x: create_point(x['Match Server'], x['Host Game Score'], x['Guest Game Score']), axis=1)

In [ ]:
swing_data_points = swing_data_points.rename(columns={'Break Point' : 'isBreakPoint'})
swing_data_points['isBreakPoint'] = swing_data_points['isBreakPoint'].replace(False, '')
swing_data_points['isBreakPoint'] = swing_data_points['isBreakPoint'].replace(True, 1)

In [ ]:
swing_data_points = swing_data_points[['Point', 'pointScore', 'isBreakPoint']]
swing_data_points

In [ ]:
swing_data = pd.merge(swing_data, swing_data_points, on='Point')

### Load in Games data

In [ ]:
your_file_name = 'EliseWagle_TexasTech_OriginalScore.xlsx'
swing_data_games = pd.read_excel(your_file_name, sheet_name='Games')
swing_data_games.shape

In [ ]:
swing_data_games.head()

In [ ]:
def create_game(player1game, player2game):
        return str(player1game) + "-" + str(player2game)
    
swing_data_games['gameScore'] = swing_data_games.apply(lambda x: create_game(x['Host Set Score'], x['Guest Set Score']), axis=1)

In [ ]:
swing_data_games.head()

In [ ]:
swing_data_games = swing_data_games[['Game', 'gameScore']]

In [ ]:
swing_data = pd.merge(swing_data, swing_data_games, on="Game")

In [ ]:
swing_data

### Create shot data csv

In [ ]:
# Check existing columns
swing_data.columns

In [ ]:
swing_data.head()

In [ ]:
# add in all desired column labels, with swingvision labels at end

columm_names = (['pointScore', 'gameScore', 'setScore', 'isPointStart', 
                                         'pointStartTime', 'isPointEnd', 'pointEndTime','pointNumber',
                                         'isBreakPoint','shotInRally','side','serverName',
                                         'serverFarNear','firstServeIn','firstServeZone',
                                         'firstServeXCoord','firstServeYCoord',
                                         'secondServeIn','secondServeZone','secondServeXCoord',
                                         'secondServeYCoord','isAce','shotContactX',
                                         'shotContactY','shotDirection','shotFhBh',
                                         'isSlice','isVolley','isOverhead','isApproach','isDropshot',
                                         'isExcitingPoint','atNetPlayer1','atNetPlayer2','isLob',
                                         'shotLocationX','shotLocationY','isWinner','isErrorWideR', 'isErrorWideL',
                                         'isErrorNet','isErrorLong','clientTeam',
                                         'Date', 'Division', 'Event', 'lineupPosition','matchDetails',
                                         'matchVenue' , 'opponentTeam', 
                                         'player1Name', 'player2Name','player1Hand','player2Hand',
                                        'Round','Surface','Notes'])

shot_data = pd.DataFrame(columns=columm_names)
shot_data

### Score Columns

In [ ]:
shot_data['pointScore'] = swing_data['pointScore']
shot_data['gameScore'] = swing_data['gameScore']

In [ ]:
shot_data.head(50)

### isPointStart and isPointEnd columns 

In [ ]:
def assign_pointstart(x):
    if (x == 'first_serve') | (x == 'second_serve'):
        return 1
    
    return ''

shot_data['isPointStart'] = swing_data['Type'].apply(assign_pointstart)


index_list = []

for i in swing_data['Point'].unique().tolist():
    last_point_index = swing_data[swing_data['Point'] == i].index[-1]
    index_list.append(last_point_index)
    
shot_data.loc[index_list,'isPointEnd'] = 1
shot_data['isPointEnd'] = shot_data['isPointEnd'].fillna('')

### pointStartTime and pointEndTime Columns

In [ ]:
# Grab the initial time
match_start_time = re.split(":", swing_data["Start Time"][0])

for i in range(len(swing_data)): # converting into milliseconds
    time = re.split(":", swing_data["Start Time"][i])
    shot_data.loc[i,'pointStartTime'] = ((int(time[0]) - int(match_start_time[0])) *  3600000
                                       + (int(time[1]) - int(match_start_time[1])) *  60000
                                       + (int(time[2]) - int(match_start_time[2])) *  1000)
    
# Assigns last shot time to pointEndTime column
shot_data['pointEndTime'] = np.where(shot_data['isPointEnd'] == 1, shot_data['pointStartTime'], '')

### pointNumber Column

In [ ]:
shot_data['pointNumber'] = swing_data['Point']

### isBreakPoint Column

In [ ]:
shot_data['isBreakPoint'] = swing_data['isBreakPoint']

### shotInRally column

In [ ]:
shot_data.shotInRally = swing_data.Shot

### side column
#### WARNING: outputs 'Center' and we don't have that metric

In [ ]:
def side(x):
    if 'deuce' in x:
        return 'Deuce'
    elif 'ad' in x:
        return 'Ad'
    elif 'center_line' in x:
        return 'Center'
    else:
        return ''

shot_data['side'] = swing_data['Hit Zone'].apply(side)

### Players

In [ ]:
# ucla roster 23-24 men and womens
ucla_roster_23 = ["Gianluca Ballotta", 
                   "Jeffrey Fradkin", 
                   "Alexander Hoogmartens",
                   "Spencer Johnson",
                   "Stefan Leustian",
                   "Timothy Li",
                   "Govind Nanda",
                   "Jorge Plans Gonzalez",
                   "Giacomo Revelli",
                   "Aadarsh Tripathi",
                   "Emon van Loben Sels",
                   "Azuma Visaya",
                   
                   "Tian Fangran",
                   "Bianca Fernandez",
                   "Ahmani Guichard",
                   "Kimmi Hance",
                   "Mia Jovic",
                   "Anne-Christine Lutkemeyer",
                   "Vanessa Ong",
                   "Sasha Vagramov",
                   "Elise Wagle"]

In [ ]:
# list of names who are playing in match
players = swing_data["Player"].unique()

# checks which one is UCLA player
is_ucla_player = [any([name in roster_name for roster_name in ucla_roster_23]) for name in players]

In [ ]:
# assigns ucla player to player 1, and non ucla to player 2
shot_data.loc[0, "player1Name"] = players[is_ucla_player]
shot_data.loc[0, "player2Name"] = players[np.invert(is_ucla_player)]

### serverName Column

In [ ]:
def assign_server_name(stroke, server):
    if stroke != 'Serve':
        return ''
    
    if server.startswith(players[is_ucla_player][0]):
        return 'Player1'
    elif server.startswith(players[np.invert(is_ucla_player)][0]):
        return 'Player2'
    
shot_data['serverName'] = swing_data.apply(lambda x: assign_server_name(x['Stroke'], x['Player']), axis=1)
shot_data['serverName'].replace(['', 'na'], pd.NaT, inplace=True)
shot_data['serverName'] = shot_data['serverName'].ffill()

### serverFarNear Column

In [ ]:
shot_data.serverFarNear = np.where((swing_data.Stroke == 'Serve'), np.where(swing_data['Hit Side'] == 'far', 'Far', 'Near'), '')
shot_data['serverFarNear'].replace(['', 'na'], pd.NaT, inplace=True)
shot_data['serverFarNear'] = shot_data['serverFarNear'].ffill()

### firstServeIn and secondServeIn Columns

In [ ]:
shot_data.firstServeIn = np.where((swing_data.Type == 'first_serve'),np.where((shot_data.isPointStart == 1) & (swing_data['Result'] == 'In'), 1, 0), np.nan)
shot_data.secondServeIn =np.where((swing_data.Type == 'second_serve') & (shot_data.isPointStart == 1), np.where(swing_data['Result'] == 'In', 1,0), np.nan)

### SwingVision Coord Transformation
court coordinates
swing vision - meters, near side center marks (0,0)
singles court x [-4.1148, 4.1148], y [0, 23.7744]
doubles court x [-5.485, 5.485]

our coordinates - center of net (0,0)
singles court x [-157.5, 157.5], y [-455, 455]

shot_x = (157.5/4.1148) * swing_x
shot_y = (455/11.8872) * swing_y + 455
ratio = 38.2764654418

### firstServeXCoord, firstYServeYCoord, secondServeXCoord, and secondServeyCoord Columns

In [ ]:
def first_serve_x_coordinates(stroke, x):
    if stroke == 'first_serve':
        return x * 38.2764654418
    else:
        return np.nan

def first_serve_y_coordinates(stroke, y):
    if stroke == 'first_serve':
        return (y - 11.8872) * 38.2764654418
    else:
        return np.nan
    
shot_data['firstServeXCoord'] = swing_data.apply(lambda row: first_serve_x_coordinates(row['Type'], row['Bounce (x)']), axis=1)
shot_data['firstServeYCoord'] = swing_data.apply(lambda row: first_serve_y_coordinates(row['Type'], row['Bounce (y)']), axis=1)


def second_serve_x_coordinates(stroke, x):
    if stroke == 'second_serve':
        return x * 38.2764654418
    else:
        return np.nan

def second_serve_y_coordinates(stroke, y):
    if stroke == 'second_serve':
        return (y - 11.8872) * 38.2764654418
    else:
        return np.nan
    
shot_data['secondServeXCoord'] = swing_data.apply(lambda row: second_serve_x_coordinates(row['Type'], row['Bounce (x)']), axis=1)
shot_data['secondServeYCoord'] = swing_data.apply(lambda row: second_serve_y_coordinates(row['Type'], row['Bounce (y)']), axis=1)

### firstServeZone and secondServeZone Columns
- serving zones: T, Body, Wide
- Wide: x in [-inf, -105] u [105, inf]
- Body: x in [-105, -52.5] u [52.5, 105]
- T: x in [-52.5, 52.5]

In [ ]:
def label_zone(x_coord):
    if x_coord != '':
        x_coord = float(x_coord)
        if (x_coord < -105) | (x_coord > 105):
            return 'Wide'
        elif (-105 <= x_coord <= -52.5) | (52.5 <= x_coord <= 105):
            return 'Body'
        elif -52.5 < x_coord < 52.5:
            return 'T'
    return ''

# convert x coord to serve zone
shot_data.firstServeZone = shot_data.firstServeXCoord.apply(label_zone)
shot_data.secondServeZone = shot_data.secondServeXCoord.apply(label_zone)

In [ ]:
shot_data.head()

### isAce Column
- WARNING: Not accurate
- FIX: counts double faults as aces

In [ ]:
shot_data['isAce'] = np.where((swing_data.Stroke == 'Serve') & 
                              (shot_data.isPointEnd == 1) & 
                              (shot_data.secondServeIn != 0), 1, np.nan)

In [ ]:
shot_data[460:600]

In [ ]:
shot_data[shot_data['isAce'] == 1]

### shotContactX and shotContactY Columns

In [ ]:
# Functions to transform the swingvision coordinates
def transf_x_coord_sv_to_shot(sv_col) :
    return sv_col * 38.2764654418
def transf_y_coord_sv_to_shot(sv_col) :
    return (sv_col - 11.8872) * 38.2764654418

# want to convert swingvision coordinates into our own
shot_data['shotContactX'] = transf_x_coord_sv_to_shot(swing_data['Hit (x)'])
shot_data['shotContactY'] = transf_y_coord_sv_to_shot(swing_data['Hit (y)'])

### shotDirection column

In [ ]:
# down the line --> switches btwn deuce and ad
# crosscourt --> remains on same side
shot_data['shotDirection'] = np.where((shot_data.shotContactX * shot_data.shotLocationX > 0) & (shot_data.shotInRally != 1), "Down the Line", 
    np.where(shot_data.shotInRally != 1, 'Crosscourt', ''))

### shotFhBh Column

In [ ]:
def classify_shot(stroke):
    if stroke == 'FH Volley':
        return 'Forehand'
    elif stroke == 'BH Volley':
        return 'Backhand'
    elif stroke == 'Forehand':
        return 'Forehand'
    elif stroke == 'Backhand':
        return 'Backhand'
    elif stroke == 'Overhead':
        return 'Forehand'
    else:
        return ''

# Applying the function to the DataFrame
shot_data['shotFhBh'] = swing_data['Stroke'].apply(classify_shot)

### isSlice, isTopspin, isFlat, isKick Columns

In [ ]:
shot_data['isSlice'] = swing_data['Spin'].apply(lambda x: '1' if x == 'Slice' else '')
shot_data['isTopspin'] = swing_data['Spin'].apply(lambda x: '1' if x == 'Topspin' else '') # added these metrics
shot_data['isFlat'] = swing_data['Spin'].apply(lambda x: '1' if x == 'Flat' else '') # added these metrics
shot_data['isKick'] = swing_data['Spin'].apply(lambda x: '1' if x == 'Kick' else '') # added these metrics

### isVolley Column
- Volleys are inaccurate

In [ ]:
shot_data['isVolley'] = swing_data['Stroke'].apply(lambda x: 1 if x in ['FH Volley', 'BH Volley'] else '') # need to classify shotFhBh when doing isVolley

In [ ]:
shot_data[shot_data['isVolley'] == 1]

In [ ]:
clean_df[clean_df['isVolley'] == 1]

### isOverhead Column
- Overheads are inaccurate

In [ ]:
shot_data['isOverhead'] = swing_data['Stroke'].apply(lambda x: 1 if x == 'Overhead' else '')

In [ ]:
shot_data[shot_data['isOverhead'] == 1]

In [ ]:
clean_df[clean_df['isOverhead'] == 1]

### isApproach Column

### isDropshot Column

### isExcitingPoint

### atNetPlayer1 and atNetPlayer2 Columns

### isLob Column

### shotLocationX and shotLocationY Columns

In [ ]:
# Functions to transform the swingvision coordinates
def transf_x_loc(stroke, sv_col):
    if stroke != 'first_serve' and stroke != 'second_serve':
        return sv_col * 38.2764654418
    return np.nan
    
def transf_y_loc(stroke, sv_col):
    if stroke != 'first_serve' and stroke != 'second_serve':
        return (sv_col - 11.8872) * 38.2764654418
    return np.nan

# want to convert swingvision coordinates into our own
shot_data['shotLocationX'] = swing_data.apply(lambda x: transf_x_loc(x['Type'], x['Bounce (x)']), axis=1)
shot_data['shotLocationY'] = swing_data.apply(lambda x: transf_y_loc(x['Type'], x['Bounce (y)']), axis=1)

### isWinner Column

In [ ]:
shot_data.isWinner = np.where((shot_data.isPointEnd == 1) & (shot_data.secondServeIn != '0') &
                              (swing_data.Result == 'In'), 1, np.nan)

In [ ]:
shot_data[shot_data['isWinner'] == 1]
# shot_data[330:345]
shot_data[shot_data['pointNumber'] == 10]

In [ ]:
clean_df[clean_df['isWinner'] == 1]
clean_df[clean_df['pointNumber'] == 10]
clean_df[40:60]

### isErrorWideR

In [ ]:
def wide_right_function(side, x, y, end):
    if (side == 'far' and x < -157.5 and y > -405 and y < 0 and end == 1) or (side == 'near' and x > 157.5 and y < 405 and y > 0 and end == 1):
        return 1
    return np.nan

# Assign 'isErrorWideR' using values from both 'swing_data' and 'shot_data'
shot_data['isErrorWideR'] = shot_data.apply(lambda x: wide_right_function(swing_data.loc[x.name, 'Hit Side'], 
                                                                x['shotLocationX'], x['shotLocationY'], x['isPointEnd']), axis=1)

### isErrorWideL

In [ ]:
def wide_left_function(side, x, y, end):
    if (side == 'far' and x > 157.5 and y > -405 and y < 0 and end == 1) or (side == 'near' and x < -157.5 and y < 405 and y > 0 and end == 1):
        return 1
    return np.nan

# Assign 'isErrorWideR' using values from both 'swing_data' and 'shot_data'
shot_data['isErrorWideL'] = shot_data.apply(lambda x: wide_left_function(swing_data.loc[x.name, 'Hit Side'], 
                                                                x['shotLocationX'], x['shotLocationY'], x['isPointEnd']), axis=1)

### isErrorNet Column

In [ ]:
shot_data.isErrorNet = np.where((swing_data.Result == 'Net'), 1, np.nan)

### isErrorLong Column

In [ ]:
shot_data['isErrorLong'] = np.where((swing_data['Result'] == 'Out') & (shot_data['shotLocationY'].abs() > 455), 1, np.nan)

In [ ]:
shot_data[shot_data['isErrorLong'] == 1]

In [ ]:
shot_data[160:175]

In [ ]:
clean_df[clean_df['isErrorLong'] == 1]

### Group First Serve and Second Serve Columns

In [ ]:
shot_data

In [ ]:
test_df = shot_data[shot_data['pointNumber'] == 3]
test_df

In [ ]:
# All columns
default_cols = ['pointScore', 'gameScore', 'setScore', 'isPointStart', 'pointStartTime',
       'isPointEnd', 'pointEndTime', 'pointNumber', 'isBreakPoint',
       'shotInRally', 'side', 'serverName', 'serverFarNear', 'firstServeIn',
       'firstServeZone', 'firstServeXCoord', 'firstServeYCoord',
       'secondServeIn', 'secondServeZone', 'secondServeXCoord',
       'secondServeYCoord', 'isAce', 'shotContactX', 'shotContactY',
       'shotDirection', 'shotFhBh', 'isSlice', 'isVolley', 'isOverhead',
       'isApproach', 'isDropshot', 'isExcitingPoint', 'atNetPlayer1',
       'atNetPlayer2', 'isLob', 'shotLocationX', 'shotLocationY', 'isWinner',
       'isErrorWideR', 'isErrorWideL', 'isErrorNet', 'isErrorLong',
       'clientTeam', 'Date', 'Division', 'Event', 'lineupPosition',
       'matchDetails', 'matchVenue', 'opponentTeam', 'player1Name',
       'player2Name', 'player1Hand', 'player2Hand', 'Round', 'Surface',
       'Notes', 'isTopspin', 'isFlat', 'isKick']

# Assign all columns to have value be taken from the first serve row
agg_dict = {col: 'first' for col in default_cols}

# Reassign select columns to have value be taken form the second serve row
agg_dict.update({'isPointEnd': 'last', 
                 'pointEndTime': 'last', 
                 'secondServeIn' : 'last',
                 'secondServeZone' : 'last',
                 'secondServeXCoord' : 'last', 
                 'secondServeYCoord' : 'last',
                 'isAce' : 'last', 
                 'shotContactX' : 'last', 
                 'shotContactY' : 'last',
                 'isWinner' : 'last',
                 'isErrorWideR' : 'last',
                 'isErrorWideL' : 'last', 
                 'isErrorNet' : 'last',
                 'isErrorLong' : 'last'
                })

# Group by isPointStart and pointNumber
new_df = test_df.groupby(['isPointStart', 'pointNumber'], as_index=False).agg(agg_dict).reset_index(drop=True)

# Identify the ungrouped rows (rows that can't be grouped)
ungrouped_rows = test_df[~test_df['shotInRally'].isin(new_df['shotInRally'])]
ungrouped_rows

test_df['shotInRally'].isin(new_df['shotInRally'])
                            
# # Concatenate grouped and ungrouped rows
# final_df = pd.concat([new_df, ungrouped_rows]).reset_index(drop=True)

# # Sort by pointNumber and shotInRally in ascending order
# final_df_sorted = final_df.sort_values(by=['pointNumber', 'shotInRally']).reset_index(drop=True)
# final_df_sorted


In [ ]:
for i in shot_data['pointNumber'].unique().tolist():

    current_point = shot_data[shot_data['pointNumber'] == i]
    serves = current_point[current_point['shotInRally'] == 1]
    serves = serves.groupby(['isPointStart', 'pointNumber'], as_index=False).agg(agg_dict).reset_index(drop=True)
    
    # Remove the serve rows from the current_point DataFrame
    current_point_no_serves = current_point[current_point['shotInRally'] != 1]
    
    # Concatenate the serves at the top of current_point
    updated_point = pd.concat([serves, current_point_no_serves]).reset_index(drop=True)


In [ ]:
# new_shot_data = shot_data.groupby(['isPointStart', 'pointNumber'], as_index=True).agg(agg_dict).reset_index(drop=True)
# new_shot_data

In [ ]:
shot_data

### Save as CSV

In [ ]:
# if want to drop swingvision coords, drops all columns after 'Notes'
# shot_data = shot_data.loc[:, :'Notes']

shot_data.to_csv('swing_vision.csv', index=False)

### Serving Columns
#### * need to revise code for determining serves in/out
- if want to hardcode it:
- far deuce serve: x [0, 157.5], y [-245, 0]
- near deuce serve: x [-157.5, 0], y [0, 245]
- far ad serve: x [-157.5, 0], y [-245, 0]
- near ad serve: x [0, 157.5], y [0, 245]

### Points

In [ ]:
# ad scoring?

ad_scoring = False

In [ ]:
# want to record the score every time a point ends
# points: server - returner
# games: ucla (player1) - opp
# sets: ucla (player1) - opp
points = np.zeros(2)
games = np.zeros(2)
sets = np.zeros(2)
pt_values = [0, 15, 30, 40]



shot_data.loc[0,"pointScore"] = f"{pt_values[int(points[0])]} - {pt_values[int(points[1])]}"
shot_data.loc[0,"gameScore"] = f"{games[0]} - {games[1]}"
shot_data.loc[0,"setScore"] = f"{sets[0]} - {sets[1]}"

shot_data["isBreakPoint"] = ''

error_cols = [x for x in shot_data.columns if "isError" in x]

for i in range(0, len(shot_data.pointScore) - 1):
    if shot_data.loc[i+1, "isPointStart"] == 1: # means we gotta update pts
        # determine point score by checking last shot
        if shot_data.loc[i, "isWinner"] == "1":
            # check if player 1 or 2 won pt
            pt_winner_player_num = (np.where(shot_data.loc[i, "Player"] == shot_data.loc[0,"player1Name"], '1', '2'))
        elif shot_data.loc[i, 'secondServeIn'] == "0": # double fault
            pt_winner_player_num = (np.where(shot_data.loc[i, "Player"] == shot_data.loc[0,"player1Name"], '2', '1'))
        elif any(shot_data.loc[i,error_cols] == "1"):
            # winner is the player who did NOT hit that shot
            pt_winner_player_num = (np.where(shot_data.loc[i, "Player"] == shot_data.loc[0,"player1Name"], '2', '1'))
        else:
            print("no pt recorded at row ", i)

        if shot_data.loc[i, "serverName"] is not None:
            didServerWinPt = shot_data.loc[i, "serverName"][-1] == pt_winner_player_num
        else:
            print(f"Server name is None at row {i}. Skipping this point.")
            continue  # Skip this point if server name is None
        
        if didServerWinPt:
            points[0] += 1
        else:
            points[1] += 1


        if ad_scoring: # checks if need to win by 2
            if any(points > 3) and abs(points[0] - points[1]) >= 2:
                game_winner = np.argmax(points)  # Find who won the game
                games[game_winner] += 1
                points = np.zeros(2)  # Reset point values
            if any(games > 5) and abs(games[0] - games[1]) >= 2:
                set_winner = np.argmax(games)  # Find who won the set
                sets[set_winner] += 1
                games = np.zeros(2)  # Reset game values   
        else:
            if points[1] == 3: # if the returner has 40 pts and can win the game
                shot_data.loc[i + 1, 'isBreakPoint'] = '1'
            if any(points > 3):
                game_winner = np.argmax(points)  # Find who won the game
                games[game_winner] += 1
                points = np.zeros(2)  # Reset point values
            if any(games > 5):
                set_winner = np.argmax(games)  # Find who won the set
                sets[set_winner] += 1
                games = np.zeros(2)  # Reset game values   

    # Update the scores in the shot_data DataFrame
    shot_data.loc[i+1,"pointScore"] = f"{pt_values[int(points[0])]} - {pt_values[int(points[1])]}"
    shot_data.loc[i+1,"gameScore"] = f"{int(games[0])} - {int(games[1])}"
    shot_data.loc[i+1,"setScore"] = f"{int(sets[0])} - {int(sets[1])}"


# Additional comments for further updates:
# - Tiebreak scenarios are not yet handled and need to be accounted for in future versions.